In [1]:
import random
import numpy as np
import tensorflow as tf

# IMAGE_SIZE = (1024, 1024)
IMAGE_SIZE = (256, 256)
# IMAGE_SIZE = (512, 512)

MASK_SIZE = IMAGE_SIZE
SEED = 7
ELA_QUALITY = 95
BATCH = 28

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

epochs = 70
INIT_LR = 3e-3

optimizer = tf.keras.optimizers.Adam(learning_rate=INIT_LR)

I0000 00:00:1779350677.133903 1135810 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779350677.407085 1135810 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779350678.642209 1135810 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779350680.235857 1135810 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device

In [2]:
import os
from PIL import Image, ImageChops, ImageEnhance
import numpy as np
from glob import glob
import cv2
import io
# from config import *

def calculate_ela(original_image, quality=ELA_QUALITY):
    with io.BytesIO() as output:
        original_image.save(output, format='JPEG', quality=quality)
        jpeg_data = output.getvalue()

    resaved_image = Image.open(io.BytesIO(jpeg_data))

    ela_image = ImageChops.difference(original_image, resaved_image)

    ela_image = ImageEnhance.Brightness(ela_image).enhance(6.0)

    return np.array(ela_image)

def load_data():
    images = list()
    masks = list()

    print('\n')
    for data_dir in data_dirs:
        image_path = path + data_dir + '/tp/*'
        mask_path = path + data_dir + '/gt/*'
        temp_images = sorted(glob(image_path))
        temp_masks = sorted(glob(mask_path))

        print("Image path:", image_path, 'Number:', len(temp_images))
        print("Mask path:", mask_path, 'Number:', len(temp_masks))
        images.extend(temp_images)
        masks.extend(temp_masks)

    print('\n')

    return images, masks

def read_images(path):
    img = Image.open(path)
    if img.mode in ('RGBA', 'LA', 'P'):
        img = img.convert('RGB')

    x1 = np.array(img)
    x1 = cv2.resize(x1, IMAGE_SIZE)
    x1 = x1 / 255
    x1 = x1.astype(np.float32)

    x2 = calculate_ela(img)
    x2 = cv2.resize(x2, IMAGE_SIZE)
    x2 = x2 / 255
    x2 = x2.astype(np.float32)

    return x1, x2

def read_mask(mask_path):
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, MASK_SIZE)
    mask = tf.cast(mask, tf.float32) / 255.0
    return mask

def preprocess_batch(X_batch, y_batch):
    def f(X, y):
        X1_batch, X2_batch, y_batch = [], [], []
        for x, m in zip(X, y):
            x1, x2 = read_images(x)
            mask = read_mask(m)
            X1_batch.append(x1)
            X2_batch.append(x2)
            y_batch.append(mask)

        return X1_batch, X2_batch, y_batch

    X1_batch, X2_batch, y_batch = tf.numpy_function(f, [X_batch, y_batch], [tf.float32, tf.float32, tf.float32])
    X1_batch.set_shape((None, *IMAGE_SIZE, 3))
    X2_batch.set_shape((None, *IMAGE_SIZE, 3))
    y_batch.set_shape((None, *IMAGE_SIZE, 1))

    return (X1_batch, X2_batch), y_batch

def tf_dataset(X, y, batch=16):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    dataset = dataset.shuffle(buffer_size=100_000, seed=SEED)
    dataset = dataset.batch(batch)
    dataset = dataset.map(preprocess_batch, num_parallel_calls=8)
    dataset = dataset.prefetch(2 * 10)

    return dataset


In [3]:
import tensorflow as tf
from keras import layers


class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        self.dense_one = layers.Dense(channels // self.ratio, activation='relu', kernel_initializer='he_normal', use_bias=True)
        self.dense_two = layers.Dense(channels, kernel_initializer='he_normal', use_bias=True)
        super().build(input_shape)

    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
        avg_pool = self.dense_two(self.dense_one(avg_pool))
        max_pool = tf.reduce_max(inputs, axis=[1, 2], keepdims=True)
        max_pool = self.dense_two(self.dense_one(max_pool))
        return inputs * tf.sigmoid(avg_pool + max_pool)

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid', kernel_initializer='he_normal', use_bias=False)
        super().build(input_shape)

    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        return inputs * self.conv(concat)

    def get_config(self):
        config = super().get_config()
        config.update({'kernel_size': self.kernel_size})
        return config


class CBAM(layers.Layer):
    def __init__(self, ratio=8, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.channel_attention = ChannelAttention(ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def call(self, inputs):
        x = self.channel_attention(inputs)
        return self.spatial_attention(x)

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio, 'kernel_size': self.kernel_size})
        return config


In [4]:
import tensorflow as tf
from keras import backend as K


def _prepare(y_true, y_pred):
    y_pred = tf.cast(tf.round(tf.cast(y_pred, tf.float32)), tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    return y_true, y_pred


def precision(y_true, y_pred):
    y_true, y_pred = _prepare(y_true, y_pred)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    return (tp + K.epsilon()) / (tp + fp + K.epsilon())


def recall(y_true, y_pred):
    y_true, y_pred = _prepare(y_true, y_pred)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    return (tp + K.epsilon()) / (tp + fn + K.epsilon())


def dice_coefficient(y_true, y_pred):
    prec = precision(y_true, y_pred)
    recal = recall(y_true, y_pred)
    return 2.0 * prec * recal / (prec + recal + K.epsilon())


def soft_dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


def dice_loss(y_true, y_pred):
    return 1.0 - soft_dice_coefficient(y_true, y_pred)


def weighted_dice_bce_loss(y_true, y_pred, weight=0.85):
    dice = dice_loss(y_true, y_pred)
    bce = tf.keras.losses.binary_crossentropy(y_true, tf.cast(y_pred, tf.float32))
    return weight * dice + (1.0 - weight) * bce


def tversky_index(y_true, y_pred, alpha=0.3, beta=0.7, smooth=1e-6):
    y_pred = tf.cast(y_pred, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    return (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)


def tversky_loss(y_true, y_pred):
    return 1.0 - tversky_index(y_true, y_pred)


def tversky_bce_loss(y_true, y_pred, weight=0.85):
    tversky = tversky_loss(y_true, y_pred)
    bce = tf.keras.losses.binary_crossentropy(y_true, tf.cast(y_pred, tf.float32))
    return weight * tversky + (1.0 - weight) * bce


def focal_tversky_loss(y_true, y_pred, gamma=0.75):
    return tf.pow(tversky_loss(y_true, y_pred), gamma)


def focal_tversky_bce_loss(y_true, y_pred, weight=0.85):
    ftl = focal_tversky_loss(y_true, y_pred)
    bce = tf.keras.losses.binary_crossentropy(y_true, tf.cast(y_pred, tf.float32))
    return weight * ftl + (1.0 - weight) * bce


def iou(y_true, y_pred):
    y_true, y_pred = _prepare(y_true, y_pred)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    return tp / (tp + fp + fn + K.epsilon())


def iou_loss(y_true, y_pred):
    return 1.0 - iou(y_true, y_pred)


def accuracy(y_true, y_pred):
    y_true, y_pred = _prepare(y_true, y_pred)
    correct = tf.reduce_sum(tf.cast(tf.equal(y_true, y_pred), tf.float32))
    total = tf.cast(tf.size(y_true), tf.float32)
    return correct / total

In [5]:
custom_objects = {
    'CBAM': CBAM,
    'dice_loss': dice_loss,
    'dice_coefficient': dice_coefficient,
    'iou': iou,
    'accuracy': accuracy,
    'tversky_bce_loss': tversky_bce_loss,
    'focal_tversky_bce_loss': focal_tversky_bce_loss,
    'weighted_dice_bce_loss': weighted_dice_bce_loss,
    'precision': precision,
    'recall': recall,
}

In [6]:
model_path = 'src/logs/ynet_new_first20may/epoch_47.keras'
model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)

/home/ubuntu/Documents/mtech/Y-Net/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/home/ubuntu/Documents/mtech/Y-Net/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'cbam_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/home/ubuntu/Documents/mtech/Y-Net/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:427: UserWarnin

In [7]:
from keras.metrics import BinaryIoU, Precision, Recall, AUC

model.compile(optimizer=optimizer, loss=focal_tversky_bce_loss, metrics=[AUC(), iou, accuracy, dice_coefficient, precision, recall], jit_compile=True)  # type: ignore[arg-type]
# model.summary()

In [8]:
path = '/media/ubuntu/New Volume1/dataset/test/'
# data_dirs = ['df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6', 'df_7']
# data_dirs = ['dcas_1_2']
data_dirs = ['dfr']

images, masks = load_data()
test_set = tf_dataset(images, masks, batch=BATCH)



Image path: /media/ubuntu/New Volume1/dataset/test/dfr/tp/* Number: 3885
Mask path: /media/ubuntu/New Volume1/dataset/test/dfr/gt/* Number: 3885




In [9]:
print('Fantastic Reality')
model.evaluate(test_set)

Fantastic Reality


I0000 00:00:1779350720.038122 1135970 service.cc:153] XLA service 0x76cc44062400 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779350720.038138 1135970 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1779350720.347660 1135970 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779350726.342431 1135970 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1779350726.416216 1135970 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_12117__.121
I0000 00:00:1779350731.511534 1135970 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set

  2/139 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.8618 - auc: 0.9378 - dice_coefficient: 0.8611 - iou: 0.7562 - loss: 0.2532 - precision: 0.7983 - recall: 0.9348   

I0000 00:00:1779350742.791570 1135970 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


138/139 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.8708 - auc: 0.9376 - dice_coefficient: 0.8690 - iou: 0.7690 - loss: 0.2435 - precision: 0.8089 - recall: 0.9396

I0000 00:00:1779350792.038977 1135971 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_12117__.121
I0000 00:00:1779350794.709458 1135971 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779350795.058570 1138006 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_95', 52 bytes spill stores, 52 bytes spill loads

I0000 00:00:1779350795.734248 1135971 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1779350796.087595 1138010 subprocess_compilation.cc:348] ptxas warning : Registers are spilled 

139/139 ━━━━━━━━━━━━━━━━━━━━ 90s 445ms/step - accuracy: 0.8725 - auc: 0.9386 - dice_coefficient: 0.8706 - iou: 0.7715 - loss: 0.2411 - precision: 0.8114 - recall: 0.9401


[0.2411062866449356,
 0.9385733604431152,
 0.7714685797691345,
 0.8724746704101562,
 0.8705902695655823,
 0.8113629221916199,
 0.9401284456253052]

In [10]:
data_dirs = ['d1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7']

images, masks = load_data()
test_set = tf_dataset(images, masks, batch=BATCH)



Image path: /media/ubuntu/New Volume1/dataset/test/d1/tp/* Number: 2153
Mask path: /media/ubuntu/New Volume1/dataset/test/d1/gt/* Number: 2153
Image path: /media/ubuntu/New Volume1/dataset/test/d2/tp/* Number: 2438
Mask path: /media/ubuntu/New Volume1/dataset/test/d2/gt/* Number: 2438
Image path: /media/ubuntu/New Volume1/dataset/test/d3/tp/* Number: 3180
Mask path: /media/ubuntu/New Volume1/dataset/test/d3/gt/* Number: 3180
Image path: /media/ubuntu/New Volume1/dataset/test/d4/tp/* Number: 3861
Mask path: /media/ubuntu/New Volume1/dataset/test/d4/gt/* Number: 3861
Image path: /media/ubuntu/New Volume1/dataset/test/d5/tp/* Number: 3554
Mask path: /media/ubuntu/New Volume1/dataset/test/d5/gt/* Number: 3554
Image path: /media/ubuntu/New Volume1/dataset/test/d6/tp/* Number: 2872
Mask path: /media/ubuntu/New Volume1/dataset/test/d6/gt/* Number: 2872
Image path: /media/ubuntu/New Volume1/dataset/test/d7/tp/* Number: 3223
Mask path: /media/ubuntu/New Volume1/dataset/test/d7/gt/* Number: 32

In [11]:
print('Defacto')
model.evaluate(test_set)

Defacto
761/761 ━━━━━━━━━━━━━━━━━━━━ 209s 272ms/step - accuracy: 0.9948 - auc: 0.9778 - dice_coefficient: 0.9536 - iou: 0.9117 - loss: 0.0726 - precision: 0.9293 - recall: 0.9795


[0.07259493321180344,
 0.9777746200561523,
 0.9117496013641357,
 0.9948012828826904,
 0.9535879492759705,
 0.9292991757392883,
 0.9795413613319397]

In [12]:
data_dirs = ['dcas_1_2']

images, masks = load_data()
test_set = tf_dataset(images, masks, batch=BATCH)



Image path: /media/ubuntu/New Volume1/dataset/test/dcas_1_2/tp/* Number: 1209
Mask path: /media/ubuntu/New Volume1/dataset/test/dcas_1_2/gt/* Number: 1209




In [13]:
model.evaluate(test_set, return_dict=True)

43/44 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.9516 - auc: 0.8665 - dice_coefficient: 0.7427 - iou: 0.5963 - loss: 0.3689 - precision: 0.8130 - recall: 0.6930

I0000 00:00:1779351161.865454 1135969 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_12117__.121


44/44 ━━━━━━━━━━━━━━━━━━━━ 19s 403ms/step - accuracy: 0.9562 - auc: 0.8685 - dice_coefficient: 0.7427 - iou: 0.5987 - loss: 0.3669 - precision: 0.8100 - recall: 0.6954


{'accuracy': 0.9561658501625061,
 'auc': 0.8684782981872559,
 'dice_coefficient': 0.7426518797874451,
 'iou': 0.5986766219139099,
 'loss': 0.36685261130332947,
 'precision': 0.8100483417510986,
 'recall': 0.6954259276390076}